# Qwen3-0.6B: IMDb Zero-Shot / Few-Shot Sentiment Classification

This notebook **does not train the model**. It uses prompts only to classify IMDb reviews with `Qwen/Qwen3-0.6B` as:

* `negative` (label 0)
* `positive` (label 1)

> Colab: GPU runtime
> Local: NVIDIA GPU or CPU


In [ ]:

# Colab / 新环境首次运行时执行。
# 安装完成后，如果 import 报版本相关错误，重启 kernel/runtime 后再从下一格开始运行。
%pip install -q -U "transformers>=4.51.0" datasets accelerate scikit-learn pandas tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.7/676.7 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatibl

In [ ]:

import os
import re
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
MODEL_NAME = "Qwen/Qwen3-0.6B"

# 设为 None：使用 Hugging Face 默认缓存目录。
# 设为文件夹，例如 "./hf_cache"：把模型和数据缓存放在项目目录下。
CACHE_DIR = None

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## 1. Download and Load the Model

There is no need to download the model manually. The first time `from_pretrained(...)` is executed, the model is automatically downloaded from Hugging Face. Subsequent runs load it from the local cache.

Qwen3 supports thinking mode by default. Since sentiment classification only requires a single label as output, thinking mode is disabled here:

```python
enable_thinking=False
```


In [ ]:

if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.float16
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    DTYPE = torch.float16
else:
    DEVICE = "cpu"
    DTYPE = torch.float32

print(f"Using device={DEVICE}, dtype={DTYPE}")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    cache_dir=CACHE_DIR,
)

tokenizer.padding_side = "left"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
    cache_dir=CACHE_DIR,
)

model.to(DEVICE)
model.eval()

print("Model loaded.")


Using device=cuda, dtype=torch.float16


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded.


## 2. Load the IMDb Dataset

The notebook first attempts to load `Kwaai/IMDB_Sentiment`, as specified in the project documentation. If that mirror cannot be loaded, it falls back to the standard `stanfordnlp/imdb` dataset on Hugging Face.

Both datasets use:

* `text`: the review text
* `label=0`: negative
* `label=1`: positive


In [ ]:

try:
    dataset = load_dataset(
        "Kwaai/IMDB_Sentiment",
        "plain_text",
        cache_dir=CACHE_DIR,
    )
    DATASET_USED = "Kwaai/IMDB_Sentiment"
except Exception as exc:
    print("项目镜像加载失败，改用 stanfordnlp/imdb。")
    print("原始错误：", repr(exc))
    dataset = load_dataset(
        "stanfordnlp/imdb",
        cache_dir=CACHE_DIR,
    )
    DATASET_USED = "stanfordnlp/imdb"

print("Dataset:", DATASET_USED)
print(dataset)
print(dataset["train"][0])


README.md:   0%|          | 0.00/4.67k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Dataset: Kwaai/IMDB_Sentiment
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In 

## 3. Text Cleaning and Truncation

IMDb reviews commonly contain `<br />` tags. Only light cleaning is applied here; stop words, punctuation, and negation words are preserved.

To speed up zero-shot and few-shot inference, long reviews are truncated by retaining the **first half and the last half**. This is generally more effective than keeping only the beginning, as the conclusion of a review often contains the reviewer’s final assessment.


In [ ]:

def clean_review(text: str) -> str:
    text = re.sub(r"<br\s*/?>", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def truncate_review(text: str, max_tokens: int = 900) -> str:
    text = clean_review(text)
    token_ids = tokenizer.encode(text, add_special_tokens=False)

    if len(token_ids) <= max_tokens:
        return text

    head_len = max_tokens // 2
    tail_len = max_tokens - head_len

    head_text = tokenizer.decode(
        token_ids[:head_len],
        skip_special_tokens=True,
    )

    tail_text = tokenizer.decode(
        token_ids[-tail_len:],
        skip_special_tokens=True,
    )

    return head_text + "\n\n[... middle content omitted ...]\n\n" + tail_text

    #shortened_ids = token_ids[:head_len] + token_ids[-tail_len:]
    #return tokenizer.decode(shortened_ids, skip_special_tokens=True)


## 4. Build the Zero-Shot and Few-Shot Prompts

The few-shot examples must come from the **training set** and must not be selected from the test set. Here, one negative and one positive training sample are selected and fixed as examples.

To prevent the model’s natural-language explanation from being mistaken for the classification label, the system prompt explicitly requires the model to output only:

```text
positive
```

or:

```text
negative
```


In [ ]:
def get_training_examples(target_label: int, n: int = 3) -> list[dict]:
    examples = []

    for example in dataset["train"]:
        if int(example["label"]) == target_label:
            examples.append({
                "text": truncate_review(example["text"], max_tokens=900),
                "label": target_label,
            })

            if len(examples) == n:
                return examples

    raise RuntimeError(f"Not enough examples for label={target_label}")


DEMO_NEG = get_training_examples(0, n=1)
DEMO_POS = get_training_examples(1, n=1)

SYSTEM_PROMPT = (
    "Classify the overall sentiment of the IMDb movie review. "
    "Reply with exactly one word: positive or negative."
)


def make_user_message(review: str) -> str:
    review = truncate_review(review, max_tokens=900)
    return f"{review}"


def build_messages(review: str, mode: str = "zero-shot") -> list[dict]:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    if mode == "few-shot":
      for demo in DEMO_NEG + DEMO_POS:
          messages.append({
              "role": "user",
              "content": make_user_message(demo["text"]),
          })
          messages.append({
              "role": "assistant",
              "content": "negative" if demo["label"] == 0 else "positive",
          })
    elif mode != "zero-shot":
          raise ValueError("mode must be 'zero-shot' or 'few-shot'")

    messages.append({"role": "user", "content": make_user_message(review)})
    return messages


## 5. Single-Sample Prediction Function

Sampling is disabled (`do_sample=False`) to make the output reproducible for the same input. Since only a single word needs to be generated, `max_new_tokens=4` is sufficient.


In [ ]:

LABEL_TEXT_TO_ID = {
    "negative": 0,
    "positive": 1,
}


def parse_label(raw_output: str):
    match = re.search(r"\b(positive|negative)\b", raw_output.lower())
    if match is None:
        return None
    return LABEL_TEXT_TO_ID[match.group(1)]


@torch.inference_mode()
def predict_sentiment(review: str, mode: str = "zero-shot") -> dict:
    messages = build_messages(review, mode=mode)
    # messages.append({
    #   "role": "user",
    #   "content": "So good, it put me to sleep for two hours.",
    # })
    # messages.append({
    #     "role": "assistant",
    #     "content": "negative",
    # })
    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=True,
        return_tensors="pt",
    )
    model_inputs = {
        key: value.to(DEVICE)
        for key, value in model_inputs.items()
    }

    start = time.perf_counter()

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=4,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

    elapsed = time.perf_counter() - start
    prompt_length = model_inputs["input_ids"].shape[1]
    new_token_ids = generated_ids[0, prompt_length:]

    raw_output = tokenizer.decode(
        new_token_ids,
        skip_special_tokens=True,
    ).strip()

    return {
        "mode": mode,
        "prediction": parse_label(raw_output),
        "raw_output": raw_output,
        "prompt_tokens": int(prompt_length),
        "seconds": elapsed,
    }

@torch.inference_mode()
def predict_sentiment_batch(
    reviews: list[str],
    mode: str = "zero-shot",
) -> list[dict]:

    prompt_texts = [
        tokenizer.apply_chat_template(
            build_messages(review, mode=mode),
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        for review in reviews
    ]

    model_inputs = tokenizer(
        prompt_texts,
        padding=True,
        return_tensors="pt",
    ).to(DEVICE)

    if torch.cuda.is_available():
      torch.cuda.synchronize()
    start = time.perf_counter()

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=4,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    if torch.cuda.is_available():
      torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    input_width = model_inputs["input_ids"].shape[1]

    output_texts = tokenizer.batch_decode(
        generated_ids[:, input_width:],
        skip_special_tokens=True,
    )

    results = []

    for i, output_text in enumerate(output_texts):
        output_text = output_text.strip()

        results.append({
            "prediction": parse_label(output_text),
            "raw_output": output_text,
            "prompt_tokens": int(
                model_inputs["attention_mask"][i].sum().item()
            ),
            "seconds": elapsed / len(reviews),
        })

    return results

## 6. Test a Single Sample First


In [ ]:

example = dataset["test"][0]

print("True label:", int(example["label"]), "(0=negative, 1=positive)")
print("Review preview:", clean_review(example["text"])[:500], "\n")

zero_result = predict_sentiment(example["text"], mode="zero-shot")
few_result = predict_sentiment(example["text"], mode="few-shot")

print("Zero-shot:", zero_result)
print("Few-shot :", few_result)


True label: 0 (0=negative, 1=positive)
Review preview: I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, CG that doesn't match the background, and painfully one-dimensional characters cannot be overcome with a 'sci-fi' setting. (I'm sure there are those of you out there who think Babylon 5 is good sci-fi  

Zero-shot: {'mode': 'zero-shot', 'prediction': 0, 'raw_output': 'negative', 'prompt_tokens': 352, 'seconds': 1.1483562029999916}
Few-shot : {'mode': 'few-shot', 'prediction': 0, 'raw_output': 'negative', 'prompt_tokens': 875, 'seconds': 0.262229829999967}


## 7. Evaluate on a Small Sample

Start by setting `N_SAMPLES` to 20 to verify that the workflow runs correctly.
In a GPU environment, gradually increase it to 100, 500, or more.

In addition to accuracy, the evaluation records:

* Valid-output coverage: whether the model produces a valid, parseable label as instructed
* Precision, recall, and F1 score
* Inference time per sample
* Number of prompt tokens


In [ ]:

def evaluate_prompt_mode(
    mode: str,
    n_samples: int = 20,
    seed: int = SEED,
    batch_size: int = 8,
) -> tuple[pd.DataFrame, dict]:
    subset = dataset["test"].shuffle(seed=seed).select(
        range(min(n_samples, len(dataset["test"])))
    )

    rows = []

    for batch_start in tqdm(
        range(0, len(subset), batch_size),
        desc=mode,
    ):
        batch_end = min(
            batch_start + batch_size,
            len(subset),
        )

        batch = subset[batch_start:batch_end]

        batch_results = predict_sentiment_batch(
            batch["text"],
            mode=mode,
        )

        for offset, result in enumerate(batch_results):
            index = batch_start + offset
            review = batch["text"][offset]
            true_label = int(batch["label"][offset])
            prediction = result["prediction"]

            rows.append({
                "index": index,
                "mode": mode,
                "true_label": true_label,
                "prediction": prediction,
                "correct": prediction == true_label,
                "raw_output": result["raw_output"],
                "prompt_tokens": result["prompt_tokens"],
                "review_preview": clean_review(review)[:2550],
                "seconds": result["seconds"],
            })

    frame = pd.DataFrame(rows)
    valid = frame["prediction"].notna()
    valid_frame = frame.loc[valid].copy()

    # 无法解析的输出在 overall accuracy 中按错误处理。
    overall_accuracy = float(frame["correct"].mean())
    coverage = float(valid.mean())

    metrics = {
        "mode": mode,
        "n_samples": len(frame),
        "coverage": coverage,
        "overall_accuracy": overall_accuracy,
        "mean_seconds_per_review": float(frame["seconds"].mean()),
        "mean_prompt_tokens": float(frame["prompt_tokens"].mean()),
    }

    if len(valid_frame) > 0:
        y_true = valid_frame["true_label"].astype(int)
        y_pred = valid_frame["prediction"].astype(int)

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            average="binary",
            zero_division=0,
        )

        metrics.update({
            "valid_only_accuracy": float(accuracy_score(y_true, y_pred)),
            "precision_positive": float(precision),
            "recall_positive": float(recall),
            "f1_positive": float(f1),
        })

    return frame, metrics


N_SAMPLES = 12000
N_SAMPLES = 25000
zero_df, zero_metrics = evaluate_prompt_mode(
    "zero-shot",
    n_samples=N_SAMPLES,
)

few_df, few_metrics = evaluate_prompt_mode(
    "few-shot",
    n_samples=N_SAMPLES,
)
#pd.DataFrame([few_metrics])
pd.DataFrame([zero_metrics, few_metrics])


zero-shot:   0%|          | 0/3125 [00:00<?, ?it/s]

few-shot:   0%|          | 0/3125 [00:00<?, ?it/s]

,mode,n_samples,coverage,overall_accuracy,mean_seconds_per_review,mean_prompt_tokens,valid_only_accuracy,precision_positive,recall_positive,f1_positive
0,zero-shot,25000,0.99320,0.83776,0.069365,316.39428,0.843496,0.774337,0.969899,0.861155
1,few-shot,25000,0.99988,0.89480,0.137414,839.39428,0.894907,0.853987,0.952716,0.900654


## 8. Review Misclassified and Unparseable Outputs


In [ ]:
import pandas as pd

pd.set_option('display.max_rows', None)       # 显示所有行
pd.set_option('display.max_columns', None)    # 显示所有列
pd.set_option('display.max_colwidth', None)   # 单元格文字不截断
pd.set_option('display.width', None)


display(
    zero_df.loc[
        (~zero_df["correct"]) | (zero_df["prediction"].isna()),
        [
            "true_label",
            "prediction",
            "raw_output",
            "prompt_tokens",
            "seconds",
            "review_preview",
        ],
    ].head(10)
)

display(
    few_df.loc[
        (~few_df["correct"]) | (few_df["prediction"].isna()),
        [
            "true_label",
            "prediction",
            "raw_output",
            "prompt_tokens",
            "seconds",
            "review_preview",
        ],
    ].head(10)
)


non_coverage_few = few_df[
    few_df["prediction"].isna()
]

display(
    non_coverage_few[
        [
            "index",
            "true_label",
            "raw_output",
            "prompt_tokens",
            "review_preview",
        ]
    ]
)


,true_label,prediction,raw_output,prompt_tokens,seconds,review_preview
4,0,1.0,positive,146,0.058340,"This movie spends most of its time preaching that it is the script that makes the movie, but apparently there was no script when they shot this waste of time! The trailer makes this out to be a comedy, but the film can't decide if it wants to be a comedy, a drama, a romance or an action film. Press releases indicated that Shatner and Hamlin made this movie because they loved the script (what were they thinking?). If you like William Shatner (I do) see ""Free Enterprise"" instead."
8,0,NaN,D,202,0.064726,"Porn legend Gregory Dark directs this cheesy horror flick that has Glen Jacobs (Kane from WWF/WWE/ whatever it calls itself nowadays) in his cinematic debut. He plays Jacob Goodknight, a blind serial killer who's forte is taking people's eyes out. The plot, be it as it may, has a group of troubled youths cleaning up the historical hotel that GoodKnight resides in and subsequently being offed by him. Hemmingway it's not. Starts of as fun dopey B-movie, but soon gets too tedious to be enjoyable. Glad I went in with pretty low expectations, but even those weren't met. How can you have a porn king directing and still suffer from a lack of nudity??? for shame. My Grade: D- Eye Candy: Samantha Noble bares her ass briefly"
12,0,1.0,positive,312,0.064726,"These days, writers, directors and producers are relying more and more on the ""surprise"" ending. The old art of bringing a movie to closure, taking all of the information we have learned through out the movie and bringing it to a nice complete ending, has been lost. Now what we have is a movie that, no matter how complex, detailed, or frivolous, can be wrapped up in 5 minutes. It was all in his/her head. That explanation is the director's safety net. If all else fails, or if the writing wasn't that good, or if we ran out of money to complete the movie, we can always say ""it was all in his/her head"" and end the movie that way. The audience will buy it because, well, none of us are psychologists, and none of us are suffering from schizophrenia (not that we know about) so we take the story and believe it. After all, the mind is a powerful thing. Some movies have pulled it off. But those movies are the reason why we are getting more and more of these crap endings. Every director/writer now thinks they can pull it off because, well, Fight Club did it and it made a lot of money. So we get movies like The Machinist, Secret Window, Identity, and this movie (just to name a few)."
13,0,1.0,positive,381,0.064726,"Teenager Tamara (Jenna Dewan) has it rough. She's ridiculed by all the popular ""kids"" for being shy, bookish, frumpy and because of her interest in witchcraft. All of the football players and cheerleaders are especially angry at her for writing an article on steroid abuse for the school paper. On top of this, her dad's a drunk, her mom's not around and she's secretly in love with her supportive English teacher (Matthew Marsden). The popular kids set up a cruel prank to humiliate her, accidentally kill her during a struggle, bury the body in the woods and make a pact to keep silent. To their shock, the very next day Tamara walks back into the classroom looking seductive and, uh, dressed to kill. Yep, back from the dead and ready for supernatural revenge against everyone involved. She also takes time out to make her sexually abusive pop eat a beer bottle and tries to seduce her English teacher away from his wife (Claudette Mink), a guidance counselor at the school. Acting and writing are strictly mediocre. You can also tell the people responsible for this have seen such movies as CARRIE (1976), HELLO MARY LOU: PROM NIGHT II (1987) and THE CRAFT (1996) because it borrows wholesale from all three of those superior films. There's some gore (a guy cutting off his ear and tongue and then stabbing his eyeball probably being the best bit), a nasty vomiting scene and a few campy/witty one-

,true_label,prediction,raw_output,prompt_tokens,seconds,review_preview
4,0,1.0,positive,669,0.100753,"This movie spends most of its time preaching that it is the script that makes the movie, but apparently there was no script when they shot this waste of time! The trailer makes this out to be a comedy, but the film can't decide if it wants to be a comedy, a drama, a romance or an action film. Press releases indicated that Shatner and Hamlin made this movie because they loved the script (what were they thinking?). If you like William Shatner (I do) see ""Free Enterprise"" instead."
12,0,1.0,positive,835,0.130664,"These days, writers, directors and producers are relying more and more on the ""surprise"" ending. The old art of bringing a movie to closure, taking all of the information we have learned through out the movie and bringing it to a nice complete ending, has been lost. Now what we have is a movie that, no matter how complex, detailed, or frivolous, can be wrapped up in 5 minutes. It was all in his/her head. That explanation is the director's safety net. If all else fails, or if the writing wasn't that good, or if we ran out of money to complete the movie, we can always say ""it was all in his/her head"" and end the movie that way. The audience will buy it because, well, none of us are psychologists, and none of us are suffering from schizophrenia (not that we know about) so we take the story and believe it. After all, the mind is a powerful thing. Some movies have pulled it off. But those movies are the reason why we are getting more and more of these crap endings. Every director/writer now thinks they can pull it off because, well, Fight Club did it and it made a lot of money. So we get movies like The Machinist, Secret Window, Identity, and this movie (just to name a few)."
17,0,1.0,positive,893,0.137309,"A holiday on a boat, a married couple, an angry waiter and a shipwreck is the reason to this films beginning. I like boobs. No question about that. But when the main character allies with whoever happens to have the most fish at the moment, mostly by having sex with them and playing the role of the constant victim, my anger just rises to a whole new level. Take two guys (a husband and another man), put a pure bombshell woman in the middle of them, ad a deserted island, subtract all her moral issues, ad a whole bunch of moral issues to the men and mix it in a big bowl of arguments, fish and a zippo lighter and you will come up with a piece of junk movie like this. The acting is, I would say, good. There are some bloopers but not many as far as i could see. The main female character makes me sick. This is due to her lack of moral values. The man with the most fish get's her attention. Even though one of them is her husband, she sees no problem with being unfaithful with (Manuel) the other man because ""I must do it to survive"". How can you justify having sex with another man for fish when your husband is 30feet away? And he won't even benefit from it? The female character has absolutely no problems to justify anything that she does. If she doesen't get approval for her actions, she's a victim. I recommend everyone to see this movie. This is the kind of movie that will make just about everything else you see this year a pleasant movie experience."
30,0,1.0,positive,995,0.111602,"Intended as light entertainment, this film is indeed successful as such during its first half, but then succumbs to a rapidly foundering script that drops it down. Harry (Judd Nelson), a ""reformed"" burglar, and Daphne (Gina Gershon), an aspiring actress, are employed as live window mannequins at a department store where one evening they are late in leaving and are locked within, whereupon they witness, from their less than protective glass observation point, an apparent homicide occurring on the street. The ostensible murderer, Miles Raymond (Nick Mancuso), a local sculptor, returns the following day to observe the mannequins since he realize

,index,true_label,raw_output,prompt_tokens,review_preview
5960,5960,1,B+,1469,"Wow, I've sure seen quite a bit of Kelli McCarty this summer. I didn't know this woman made so many softcore flicks in the past three years. It's like seeing a future softcore star blossom in front of me, much like Michelle Hall did a couple of years ago. ""Passion's Peak"" is the third quality softcore flick I've seen Kelli McCarty in, with ""Girl for Girl"" and ""House of Love"" being the others. ""Desire and Deception"" was okay, but it wasn't spectacular. There's spoilers in this review, so read only if you want to. The story begins with Christina (Kelli McCarty) heading out of the big city and to the mountains. She has inherited a house from her dear departed grandmother and plans to turn it into a mountain lodge. Before she can even set her things down, some woman named Kim (uncredited in this film, but quite the aggressive one) begins booking guests to stay there. Now she has to get the house into shape quickly--in comes Chip (Bobby Johnston), a childhood friend, to the rescue. Chip helps her get the house in workable condition. She hires two local slackers to work in her lodge--Chip's sister Bait (Samantha McConnell) and her sex buddy Hank. Now the guests start coming. The first to arrive are Eric and Linda (Flower), two stereotypical money-first lawyers. Linda and Eric get into a huge argument during a dinner party halfway through the film which leads to their breakup; sad stuff there. Next, there's romance novelist Sophia, played by B-movie goddess Monique Parent. She's using that silly alias Scarlet Johansing again, and she's got a very professional look this time--with blonde hair, of course. It wouldn't be Monique if she didn't have at least one scene where she plays with herself--and she obliges, during one of Eric and Linda's sex scenes. James and Shene (Devinn Lane--yes, the porn star Devinn Lane) show up for a little weekend getaway as well. Unbeknownst to Shene, James and Christina have quite a history. James and Christina used to date, but Christina broke it off to head to this mountain lodge. James comes up to the lodge to get Christina back, but his plan backfires. Christina spills the beans to Shene, which causes Shene to walk out on him and down to the local bar to strip for the locals. Shene ends up in the sleeping bag of the now-single Eric, and they leave together. Bait realizes she wants something more than just sex with Hank, and Christine finds true love with Chip, with Sophia soaking it all in and writing it into her next romance novel. In fact....if you ask me, this whole movie play"
10251,10251,0,Boring,611,"Can you say ""Boring"" with a capital B! It's slower than watching grass grow! It's more boring than watching paint dry! You'll sleep right through it.....we all did.....don't do it...you'll regret it!"
18576,18576,0,bad,1017,"**Might contain spoilers** Ok, lets conclude this movie in one word: bad. Two words? Really bad. Now why do I think that? Let me explain. Guttenberg leads a special-ops team consisting of four persons that get assigned to retake an lethal virus after some arms-dealer stole it from a lab. They do this by attacking the arm-dealers in mid-flight and somehow gets back the virus after some fighting. However, suspicions arise about Guttenberg because one of the terrorist knew his name. After debriefing the team-members get attacked by unknown persons and everyone starts to suspect everyone else is involved. After deciding they cant trust their bosses, they decide to, once again, steal the virus and try to lure out the possible attackers. In theory this is a plot that could have worked in a low-budget movie that just aims to be aired on TV. However, the plot is compromised and the movie ruined in several accounts. Firstly, the plot is totally predictable and it is not fun to know how the movie is going to end after three minutes. Second. The acting is really bad, or the actors are directed to act a

## 9. Confusion Matrix and Detailed Classification Report


In [ ]:

def print_valid_classification_report(frame: pd.DataFrame) -> None:
    valid = frame["prediction"].notna()
    valid_frame = frame.loc[valid]

    if len(valid_frame) == 0:
        print("没有可解析的预测。")
        return

    y_true = valid_frame["true_label"].astype(int)
    y_pred = valid_frame["prediction"].astype(int)

    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print()
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=["negative", "positive"],
            digits=4,
            zero_division=0,
        )
    )


print("ZERO-SHOT")
print_valid_classification_report(zero_df)

print("\nFEW-SHOT")
print_valid_classification_report(few_df)


ZERO-SHOT
Confusion matrix:
[[ 8893  3512]
 [  374 12051]]

              precision    recall  f1-score   support

    negative     0.9596    0.7169    0.8207     12405
    positive     0.7743    0.9699    0.8612     12425

    accuracy                         0.8435     24830
   macro avg     0.8670    0.8434    0.8409     24830
weighted avg     0.8669    0.8435    0.8409     24830


FEW-SHOT
Confusion matrix:
[[10462  2036]
 [  591 11908]]

              precision    recall  f1-score   support

    negative     0.9465    0.8371    0.8885     12498
    positive     0.8540    0.9527    0.9007     12499

    accuracy                         0.8949     24997
   macro avg     0.9003    0.8949    0.8946     24997
weighted avg     0.9003    0.8949    0.8946     24997



## 10. Save the Results for Reporting and Error Analysis


In [ ]:

OUTPUT_DIR = Path("./qwen3_prompt_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

zero_df.to_csv(OUTPUT_DIR / "zero_shot_predictions.csv", index=False)
few_df.to_csv(OUTPUT_DIR / "few_shot_predictions.csv", index=False)

summary_df = pd.DataFrame([zero_metrics, few_metrics])
summary_df.to_csv(OUTPUT_DIR / "summary_metrics.csv", index=False)

print("Saved to:", OUTPUT_DIR.resolve())
display(summary_df)


Saved to: /content/qwen3_prompt_results


,mode,n_samples,coverage,overall_accuracy,mean_seconds_per_review,mean_prompt_tokens,valid_only_accuracy,precision_positive,recall_positive,f1_positive
0,zero-shot,25000,0.99320,0.83776,0.069365,316.39428,0.843496,0.774337,0.969899,0.861155
1,few-shot,25000,0.99988,0.89480,0.137414,839.39428,0.894907,0.853987,0.952716,0.900654
